# Telugu OCR — Colab + EasyOCR (GPU), checkpointed
Manual-collection source for the 20% requirement. A book's text is only written to the output file after it finishes OCR completely, so a crash mid-book never leaves duplicate or partial content in the corpus — the whole book just gets retried next run.

In [ ]:
!pip install -q pymupdf easyocr tqdm

In [ ]:
import os
import json
import pymupdf
import easyocr
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define directory paths (update INPUT_FOLDER if your PDFs are stored elsewhere)
INPUT_FOLDER = '/content/drive/MyDrive/LMA DATASET 2'
OUTPUT_DIR = '/content/drive/MyDrive/LMA DATASET 2/ocr_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. File paths for storage, checkpoints, and stats
OUTPUT_TEXT_FILE = os.path.join(OUTPUT_DIR, 'raw_manual_ocr.txt')
CHECKPOINT_FILE  = os.path.join(OUTPUT_DIR, 'processed_pdfs.json')
STATS_FILE       = os.path.join(OUTPUT_DIR, 'per_book_stats.json')

# 4. Initialize EasyOCR with GPU acceleration
print("Loading EasyOCR Telugu Model on GPU...")
reader = easyocr.Reader(['te', 'en'], gpu=True)
print("EasyOCR loaded successfully!")

In [ ]:
def load_checkpoint():
    """Loads the set of successfully processed PDF filenames."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_checkpoint(done_set):
    """Saves the completed PDF filenames list to Google Drive."""
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(sorted(done_set), f, ensure_ascii=False, indent=2)

def load_stats():
    """Loads extracted stats per processed PDF."""
    if os.path.exists(STATS_FILE):
        with open(STATS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_stats(stats):
    """Saves updated stats per PDF to Google Drive."""
    with open(STATS_FILE, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

In [ ]:
def process_pdfs_in_folder(folder_path, output_path):
    print(f"Scanning folder: {folder_path}")
    if not os.path.exists(folder_path):
        print(f"Error: Folder does not exist at {folder_path}")
        return

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print("No PDF files found in the specified folder.")
        return

    done = load_checkpoint()
    stats = load_stats()
    todo = [f for f in pdf_files if f not in done]

    print(f"Found {len(pdf_files)} PDF(s) total | {len(done)} completed | {len(todo)} remaining.\n")

    for pdf_file in todo:
        file_path = os.path.join(folder_path, pdf_file)
        print(f"--- Processing: {pdf_file} ---")

        try:
            doc = pymupdf.open(file_path)
        except Exception as e:
            print(f"  Could not open {pdf_file}, skipping: {e}")
            done.add(pdf_file)
            save_checkpoint(done)
            continue

        total_pages = len(doc)
        book_extracted_pages = []
        doc_words = 0
        page_num = -1  # guards against NameError in the except block if total_pages == 0

        try:
            for page_num in range(total_pages):
                page = doc[page_num]
                # Render page image at 2x resolution for clear OCR
                matrix = pymupdf.Matrix(2.0, 2.0)
                pix = page.get_pixmap(matrix=matrix, alpha=False)

                # Convert raw image buffer into numpy array
                img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

                # Run EasyOCR
                result = reader.readtext(img_array, detail=0)
                extracted_text = " ".join(result).strip()
                if extracted_text:
                    book_extracted_pages.append(extracted_text)
                    doc_words += len(extracted_text.split())

                if (page_num + 1) % 10 == 0 or (page_num + 1) == total_pages:
                    print(f"  Page {page_num + 1}/{total_pages} processed.")

            # Append complete book to output file ONLY after successful processing
            with open(output_path, 'a', encoding='utf-8') as out_file:
                out_file.write(f"<<SRC:ocr_manual>><<FILE:{pdf_file}>>\n")
                out_file.write("\n".join(book_extracted_pages) + "\n<<END>>\n\n")
            doc.close()

            # Mark complete and save state to Drive
            done.add(pdf_file)
            stats[pdf_file] = {"pages": total_pages, "words": doc_words}
            save_checkpoint(done)
            save_stats(stats)

            print(f"Finished {pdf_file}. Words extracted: {doc_words:,}\n")

        except Exception as e:
            print(f"  Error while processing {pdf_file} on page {page_num+1}: {e}")
            print(f"  Book NOT marked as complete; skipped safely so it can retry next run.")
            doc.close()
            continue

    # Summary Statistics
    total_words_all = sum(s["words"] for s in stats.values())
    print("=== Batch OCR Complete ===")
    print(f"Total books processed so far: {len(stats)}")
    print(f"Total words extracted across all books: {total_words_all:,}")
    print(f"Estimated Telugu tokens (~1.7x factor): {int(total_words_all * 1.7):,}")
    print(f"Output text saved to: {output_path}")

# Run OCR batch processing
process_pdfs_in_folder(INPUT_FOLDER, OUTPUT_TEXT_FILE)